# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [8]:
# Escribe tu código aquí para limpiar y enriquecer los datos
import pandas as pd
from pathlib import Path
import numpy as np

In [5]:
data_dir = Path('../data/raw')

In [6]:
df = pd.read_csv(data_dir / 'housing' / 'housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [16]:
df = df.dropna()

In [17]:
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['persons_per_house'] = df['population'] / df['households']
df['income_per_person'] = df['median_income'] / df['population']
df['rooms_per_person'] = df['total_rooms'] / df['population']
df['bedrooms_per_household'] = df['total_bedrooms'] / df['households']


In [18]:
def haversine(lon1, lat1, lon2, lat2):
  # Convert to radians
  lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
  dlon = lon2 - lon1
  dlat = lat2 - lat1
  a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
  c = 2 * np.arcsin(np.sqrt(a))
  R = 6371  # Earth radius in km
  return R * c

In [19]:
lon_point = -122.51098595945538
lat_point = 37.758591813358514

df['distance_to_beach'] = haversine(
    df['longitude'],
    df['latitude'],
    lon_point,
    lat_point
)

In [20]:
df1 = df.drop('ocean_proximity', axis=1)
df1.corr()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,rooms_per_household,bedrooms_per_room,persons_per_house,income_per_person,rooms_per_person,bedrooms_per_household,distance_to_point,distance_to_beach
longitude,1.000000,-0.924616,-0.109357,0.045480,0.069608,0.100270,0.056513,-0.015550,-0.045398,-0.027307,0.092657,0.002304,-0.034758,-0.073690,0.013402,0.965295,0.956206
latitude,-0.924616,1.000000,0.011899,-0.036667,-0.066983,-0.108997,-0.071774,-0.079626,-0.144638,0.106423,-0.113815,0.002522,0.028240,0.140037,0.070025,-0.899612,-0.897458
housing_median_age,-0.109357,0.011899,1.000000,-0.360628,-0.320451,-0.295787,-0.302768,-0.118278,0.106432,-0.153031,0.136089,0.013258,0.020326,-0.107854,-0.077918,-0.107660,-0.103104
total_rooms,0.045480,-0.036667,-0.360628,1.000000,0.930380,0.857281,0.918992,0.197882,0.133294,0.133482,-0.187900,-0.024596,-0.139661,0.128937,0.029373,0.035090,0.033734
total_bedrooms,0.069608,-0.066983,-0.320451,0.930380,1.000000,0.877747,0.979728,-0.007723,0.049686,0.001538,0.084238,-0.028355,-0.164081,0.056592,0.045887,0.062203,0.061395
population,0.100270,-0.108997,-0.295787,0.857281,0.877747,1.000000,0.907186,0.005087,-0.025300,-0.071898,0.035319,0.070062,-0.170879,-0.140144,-0.066510,0.090003,0.089073
households,0.056513,-0.071774,-0.302768,0.918992,0.979728,0.907186,1.000000,0.013434,0.064894,-0.080165,0.065087,-0.027336,-0.170398,-0.028413,-0.055158,0.050780,0.051226
median_income,-0.015550,-0.079626,-0.118278,0.197882,-0.007723,0.005087,0.013434,1.000000,0.688355,0.325307,-0.615661,0.018894,0.212246,0.236566,-0.062299,-0.027273,-0.023656
median_house_value,-0.045398,-0.144638,0.106432,0.133294,0.049686,-0.025300,0.064894,0.688355,1.000000,0.151344,-0.255880,-0.023639,0.114895,0.208544,-0.046739,-0.042280,-0.031836
rooms_per_household,-0.027307,0.106423,-0.153031,0.133482,0.001538,-0.071898,-0.080165,0.325307,0.151344,1.000000,-0.416952,-0.004873,0.141805,0.887736,0.848616,-0.041181,-0.046139
